In [2]:
from dotenv import load_dotenv,find_dotenv
import os
from langchain.embeddings import OpenAIEmbeddings
import pinecone  #interact with pinecone

from langchain_community.vectorstores import Pinecone  #interact with langchain pinecone module

from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import WikipediaLoader


load_dotenv(find_dotenv(),override=True)
# Have the below keys in .env file
# OPENAI_API_KEY
# PINECONE_API_KEY

import re

from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader



def bs4_extractor(html: str) -> str:
    soup = BeautifulSoup(html, "lxml")
    return re.sub(r"\n\n+", "\n\n", soup.text).strip()

# since the links in webpage are showing docs/location etc , i think it is not picking the other links
loader = RecursiveUrlLoader("https://github.com/langchain-ai/langchain", extractor=bs4_extractor)
docs = loader.load()
# print(docs[0].page_content[:200])

# 

# from langchain_community.document_loaders.sitemap import SitemapLoader
# import nest_asyncio

# nest_asyncio.apply()
# # sitemap_loader = SitemapLoader(web_path="https://python.langchain.com/sitemap.xml")
# loader = SitemapLoader(
#     web_path="https://python.langchain.com/sitemap.xml",
#     filter_urls=["https://python.langchain.com/docs/integrations/providers/"]
# )

# docs = loader.load()

pc = pinecone.Pinecone()


# indexes = pc.list_indexes().names()
# for i in indexes:
#     print('Deleting all indexes ... ', end='')
#     pc.delete_index(i)
#     print('Done')

# from pinecone import ServerlessSpec

index_name = 'langchain-webpage'
# if index_name not in pc.list_indexes().names():
#     print(f'Creating index {index_name}')
#     pc.create_index(
#         name=index_name,
#         dimension=1536,
#         metric='cosine',
#         spec=ServerlessSpec(
#             cloud="aws",
#             region="us-east-1"
#         ) 
#     )
#     print('Index created! 😊')
# else:
#     print(f'Index {index_name} already exists!')


In [3]:
len(docs)

1

In [ ]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-large', dimensions=1536)  # 512 works as well


#only included 10 docs of the webpage due to limit in metadta size of pinecone account
vector_store = Pinecone.from_documents(docs[0:10], embeddings, index_name=index_name)  #created vecotr in pinecone, which can be loaded when needed.


In [ ]:
vector_store_load = Pinecone.from_existing_index(index_name=index_name,embedding=embeddings)


In [ ]:

# Initialize the LLM with the specified model and temperature
llm = ChatOpenAI(model='gpt-3.5-turbo', temperature=0.2)

# Use the provided vector store with similarity search and retrieve top k results
retriever = vector_store_load.as_retriever(search_type='similarity', search_kwargs={'k': 1})

# Create a RetrievalQA chain using the defined LLM, chain type 'stuff', and retriever
chain = RetrievalQA.from_chain_type(llm=llm, chain_type='stuff', retriever=retriever)

query = 'can you tell what all do you know about integrations related to langchain. Also list Top 5 providers'
answer = chain.invoke(query)
print(answer['result'])